# Delta Lakehouse demo (read-only)

NYC Yellow Taxi 2025 through Bronze -> Silver -> Gold on Delta Lake 2.4 / Spark 3.4.

* Every cell only **reads** tables and `_delta_log` files; nothing is written, restored or vacuumed.
* Versions are discovered from Delta history, never hard-coded.
* Each cell should finish in under ~20 seconds on the demo laptop (8 cores).

Run cells top to bottom. Section 0 starts Spark (~15 s).

## 0. Setup

In [ ]:
import json
import sys
import time
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():  # started from notebooks/
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from delta.tables import DeltaTable
from pyspark.sql import functions as F

from optimization_benchmark import QUERIES, ZORDER_COLUMN, find_optimize_versions
from src.audit.time_travel import (
    commit_action_preview,
    file_scoped_insert,
    file_scoped_update,
    find_column_introduction,
    inspect_delta_log_commit,
    metadata_commit_versions,
    normalize_history,
    select_audit_versions,
)
from src.common.config import (
    BATCH_AUDIT_PATH, BRONZE_PATH, EVIDENCE_DIR, GOLD_PATH, REJECTED_PATH, SILVER_PATH,
)
from src.common.spark import create_spark
from src.optimization.data_skipping import active_files, column_range, files_to_read

spark = create_spark("lakehouse-demo")
spark.sparkContext.setLogLevel("ERROR")
BRONZE, SILVER = str(BRONZE_PATH), str(SILVER_PATH)


def history(path):
    """Delta history as a DataFrame with the most useful metrics flattened."""
    m = "operationMetrics"
    return (
        DeltaTable.forPath(spark, path).history()
        .select(
            "version", "timestamp", "operation",
            F.col(f"{m}.numOutputRows").alias("output_rows"),
            F.col(f"{m}.numSourceRows").alias("source_rows"),
            F.col(f"{m}.numTargetRowsUpdated").alias("updated"),
            F.col(f"{m}.numTargetRowsInserted").alias("inserted"),
            F.col(f"{m}.numTargetRowsCopied").alias("copied"),
        )
        .orderBy("version")
    )


print("Spark", spark.version, "| Silver latest version:",
      DeltaTable.forPath(spark, SILVER).history(1).first()["version"])

## 1. Bronze - append-only raw layer

One Delta commit per input batch: 12 monthly TLC files + the dirty fixture, then the CDC demo events.

In [ ]:
history(BRONZE).select("version", "timestamp", "operation", "output_rows").show(30, truncate=False)

The dirty fixture batch keeps every problem Silver must handle (malformed dates, NULL zones/GPS, negative fares, duplicates):

In [ ]:
dirty = spark.read.format("delta").load(BRONZE).filter(F.col("ingest_batch_id") == "dirty_test")
dirty.agg(
    F.count("*").alias("rows"),
    F.sum((F.col("tpep_pickup_datetime") == "not-a-date").cast("int")).alias("bad_pickup"),
    F.sum(F.col("tpep_dropoff_datetime").startswith("2025-99").cast("int")).alias("bad_dropoff"),
    F.sum(F.col("PULocationID").isNull().cast("int")).alias("null_pu_zone"),
    F.sum(F.col("DOLocationID").isNull().cast("int")).alias("null_do_zone"),
    F.sum(F.col("pickup_latitude").isNull().cast("int")).alias("null_pickup_gps"),
    F.sum((F.col("fare_amount") < 0).cast("int")).alias("negative_fare"),
    (F.count("*") - F.countDistinct("raw_record_hash")).alias("duplicate_rows"),
).show()
(dirty.filter((F.col("tpep_pickup_datetime") == "not-a-date") | (F.col("fare_amount") < 0))
 .select("tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "fare_amount", "record_source")
 .show(5, truncate=False))

Bronze version 0 in `_delta_log`: the actions of the first commit and the PULocationID range from its per-file statistics.

In [ ]:
v0 = inspect_delta_log_commit(BRONZE, 0, sample_files=2)
print({k: v0[k] for k in ("operation", "add_count", "remove_count", "protocol")})
print("first columns:", v0["metadata_columns"][:8], "...")
ranges = [column_range(stats, "PULocationID") for stats in active_files(BRONZE, 0).values()]
ranges = [r for r in ranges if r]
print("PULocationID min/max over v0 files:", min(r[0] for r in ranges), "-", max(r[1] for r in ranges))

## 2. Silver - quality rules, dedup and per-microbatch audit

In [ ]:
audit = spark.read.format("delta").load(str(BATCH_AUDIT_PATH))
audit.agg(
    F.count("*").alias("microbatches"),
    F.sum("incoming").alias("incoming"),
    F.sum("valid").alias("valid"),
    F.sum("rejected").alias("rejected"),
    F.sum("winners").alias("winners"),
).show()
print("Silver rows now:", f"{spark.read.format('delta').load(SILVER).count():,}")

In [ ]:
rejected = spark.read.format("delta").load(str(REJECTED_PATH))
(rejected.select(F.explode("rejection_reasons").alias("reason"))
 .groupBy("reason").count().orderBy(F.desc("count")).show(truncate=False))

## 3. CDC with Delta time travel

Find the UPDATE, INSERT and schema-evolution commits from history and metrics, then read the table *as of* the version before and after each one.

In [ ]:
rows = normalize_history(DeltaTable.forPath(spark, SILVER).history(200).collect())
selected = select_audit_versions(rows, metadata_commit_versions(SILVER))
UPDATE_V = selected["update_after_version"]
INSERT_V = selected["insert_after_version"]
SCHEMA_V = find_column_introduction(SILVER, "surcharge_fee")["version"]
print(f"UPDATE v{UPDATE_V}, INSERT v{INSERT_V}, SCHEMA v{SCHEMA_V}")
history(SILVER).filter(F.col("version") >= UPDATE_V - 1).show(truncate=False)

In [ ]:
# UPDATE: same trip_id, fare/tip changed. The changed row comes from the files the commit rewrote.
update = file_scoped_update(spark, SILVER, UPDATE_V)[0]
TRIP = update["trip_id"]
for v in (UPDATE_V - 1, UPDATE_V):
    print(f"versionAsOf {v}")
    (spark.read.format("delta").option("versionAsOf", v).load(SILVER)
     .filter(F.col("trip_id") == TRIP)
     .select("trip_id", "fare_amount", "tip_amount", "record_source", "ingest_batch_id")
     .show(truncate=False))

In [ ]:
# INSERT: a new trip_id appears and the row count grows by one.
insert = file_scoped_insert(spark, SILVER, INSERT_V)[0]
for v in (INSERT_V - 1, INSERT_V):
    snapshot = spark.read.format("delta").option("versionAsOf", v).load(SILVER)
    found = snapshot.filter(F.col("trip_id") == insert["trip_id"]).count()
    print(f"versionAsOf {v}: {snapshot.count():,} rows, new trip present: {found}")

In [ ]:
# SCHEMA: surcharge_fee exists only from the schema commit on.
for v in (SCHEMA_V - 1, SCHEMA_V):
    snapshot = spark.read.format("delta").option("versionAsOf", v).load(SILVER)
    has = "surcharge_fee" in snapshot.columns
    print(f"versionAsOf {v}: surcharge_fee column = {has}")
    if has:
        (snapshot.filter(F.col("trip_id") == TRIP)
         .select("trip_id", "fare_amount", "tip_amount", "surcharge_fee").show(truncate=False))

In [ ]:
# The syntax from the assignment: read the very first version of Silver.
first = spark.read.format("delta").option("versionAsOf", 0).load(SILVER)
print(f"versionAsOf 0: {first.count():,} rows, {len(first.columns)} columns")

## 4. What the `_delta_log` commits contain

In [ ]:
print(f"{'version':>7} {'operation':<10} {'add':>4} {'remove':>6} {'metaData':>8}  key metrics")
for v in (UPDATE_V, INSERT_V, SCHEMA_V):
    c = inspect_delta_log_commit(SILVER, v)
    m = c["operationMetrics"]
    keys = ("numTargetRowsUpdated", "numTargetRowsInserted", "numTargetRowsCopied")
    print(f"{v:>7} {c['operation']:<10} {c['add_count']:>4} {c['remove_count']:>6} "
          f"{str(c['metadata_columns'] is not None):>8}  " + ", ".join(f"{k}={m.get(k)}" for k in keys))

In [ ]:
# The new metaData action: the schema now ends with surcharge_fee.
for line in commit_action_preview(SILVER, SCHEMA_V, max_chars=160):
    print(line)
schema_commit = inspect_delta_log_commit(SILVER, SCHEMA_V)
print("last columns of the new schema:", schema_commit["metadata_columns"][-4:])

## 5. OPTIMIZE and Z-ORDER - data skipping

States of the same table, kept side by side by time travel: A = baseline (before OPTIMIZE), B = compaction, C = Z-ORDER BY PULocationID.

In [ ]:
compaction_v, zorder_v = find_optimize_versions(DeltaTable.forPath(spark, SILVER))
STATES = {"A_baseline": compaction_v - 1, "B_compaction": compaction_v, "C_zorder": zorder_v}
print(STATES)
for state, v in STATES.items():
    files = active_files(SILVER, v)
    widths = [r[1] - r[0] for r in (column_range(s, ZORDER_COLUMN) for s in files.values()) if r]
    print(f"{state:<13} v{v}: {len(files):>4} files, avg PULocationID range per file "
          f"{sum(widths) / len(widths):6.1f}, files to read for zone 132: "
          f"{files_to_read(files, ZORDER_COLUMN, (132, 132))}")

In [ ]:
# Live Q1 (zone 132 = JFK) on the baseline and the Z-ORDERed version; one warm-up run each.
_, _, q1 = QUERIES["Q1_single_zone"]
for state in ("A_baseline", "C_zorder"):
    df = spark.read.format("delta").option("versionAsOf", STATES[state]).load(SILVER)
    q1(df).collect()
    start = time.perf_counter()
    result = q1(df).collect()[0]
    print(f"{state:<11} {time.perf_counter() - start:5.2f} s  trips={result[0]:,}")

In [ ]:
# Recorded benchmark (1 warm-up + 5 measured runs, median) from docs/evidence.
evidence = json.loads(sorted(EVIDENCE_DIR.glob("task4_benchmark_*.json"))[-1].read_text())
print(f"{'state':<14}{'query':<17}{'files':>9}{'median s':>10}{'speedup':>9}")
for r in evidence["results"]:
    files = f"{r['files_to_read']}/{r['files_total']}"
    print(f"{r['state']:<14}{r['query']:<17}{files:>9}{r['median_s']:>10.3f}{r['speedup_vs_A']:>8.2f}x")

## 6. Gold - top zone x hour groups by driver earnings

In [ ]:
gold = spark.read.format("delta").load(str(GOLD_PATH))
(gold.orderBy(F.desc("total_driver_earnings"))
 .select("PULocationID", "pickup_hour", "trip_count",
         F.round("avg_fare", 2).alias("avg_fare"),
         F.round("tip_pct", 1).alias("tip_pct"),
         F.round("avg_driver_earnings", 2).alias("avg_earnings"),
         F.round("total_driver_earnings").cast("long").alias("total_earnings"),
         "silver_version")
 .show(5))

In [ ]:
spark.stop()